In [48]:
from transformers import TrainerCallback
from lightning.pytorch.callbacks import Callback
import os
import json
import torch
import numpy as np
import wandb
import pandas as pd
from datetime import datetime
from utils.countdown_utils import *
from tqdm import trange
from pathlib import Path
from transformers import AutoConfig, AutoModelForCausalLM
from litgpt.scripts.convert_lit_checkpoint import convert_lit_checkpoint
from litgpt.utils import copy_config_files, auto_download_checkpoint
from transformers import AutoTokenizer, PreTrainedTokenizerFast

%load_ext autoreload
%autoreload 2
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
def eval_ll(
        model,
        tokenizer,
        data,
        batch_size=128,
        context_len=4096,
        temperature=0.0,
        n=1,
    ):
        """
        Evaluate the model on the data using a sliding window so that the context length is not exceeded
        """
        output_texts_concat = []
        for b in trange(0, len(data), batch_size):
            batch = data[b : min(b + batch_size, len(data))]
            output_texts = ["" for _ in range(len(batch))]
            tokenizer.padding_side = "left"
            inputs = tokenizer(batch, return_tensors="pt", padding=True).to("cuda")
            inputs = inputs["input_ids"]

            if n == 1:
                outputs = model.generate(
                    input_ids=inputs,
                    pad_token_id=tokenizer.eos_token_id,
                    attention_mask=torch.ones_like(inputs),
                    max_length=context_len,
                    num_beams=1,
                    do_sample=False,
                )
                output_tokens = outputs
                output_text = tokenizer.batch_decode(
                    output_tokens, skip_special_tokens=False
                )
                tokenizer.padding_side = "left"
                output_texts = [
                    ot + ot_now for ot, ot_now in zip(output_texts, output_text)
                ]
                output_texts_concat += output_texts

        return output_texts_concat

In [57]:
data_file = "data/generalization_data/test1_b3_t30_n1000_dfs_filtered.json"
# data_file = "data/sos_filtered/test1_b4_t30_n2000000_dfs_filtered.json"

data = []

with open(data_file, "r") as f:
    for line in f:
        if line.strip():  # Skip empty lines
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Error parsing line: {e}")
                continue

In [58]:
model_dir = Path("temp/hf_Pythia-6-4-64-dfs-cosine-2")
state_dict = torch.load(model_dir / "model.pth")

hf_model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype=torch.bfloat16,
        local_files_only=True,
        state_dict=state_dict,
        attn_implementation="flash_attention_2",
    )

hf_model.cuda()
hf_model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)

/tmp/ipykernel_504869/311978829.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_dir / "model.pth")


In [59]:
test_prompts = [
    tokenizer.bos_token
    + f"S {sample['target']} [ {' '.join(map(str,sample['nums']))} ] ,"
    for sample in data[: 16]
    ]

In [60]:
predictions = eval_ll(
                hf_model,
                tokenizer,
                data=test_prompts,
                batch_size=16,
                context_len=4096,
                temperature=0.0,
                n=1,
            )

100%|██████████| 1/1 [00:21<00:00, 21.78s/it]


In [61]:
pred_ratings = []
true_rating = []
pred_reasons = []

for i in range(len(predictions)):
    rating, reason = metric_fn(
        predictions[i]
        .split(tokenizer.bos_token)[1]
        .split(tokenizer.eos_token)[0],
        mode="sft",
    )
    tr, _ = metric_fn(f"{data[i]['search_path']}", mode="sft")
    pred_ratings.append(rating)
    true_rating.append(tr)
    pred_reasons.append(reason)

pred_ratings = np.array(pred_ratings)
avg_rating = float(np.mean(pred_ratings))
avg_true_rating = float(np.mean(true_rating))
accuracy = float(np.mean([r > 0 for r in pred_ratings]))
true_accuracy = float(np.mean([r > 0 for r in true_rating]))

In [62]:
pred_ratings

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [63]:
print(f"Average rating: {avg_rating}")
print(f"-- Average true rating: {avg_true_rating}")
print(f"Accuracy: {accuracy}")
print(f"-- True accuracy: {true_accuracy}")

Average rating: 0.0
-- Average true rating: 0.7434895833333334
Accuracy: 0.0
-- True accuracy: 1.0


In [67]:
predictions[8]

'[BOS] S 15 [ 13 5 16 ] , E 5 + 5 = 22 R [ 13 16 16 ] , G #00 15 [ 13 16 22 ] , M #00 , S 15 [ 13 16 22 ] , E 22 - 13 = 9 R [ 16 9 ] , G #000 15 [ 16 9 ] , M #000 , S 15 [ 16 9 ] , E 16 - 9 = 7 R [ 7 ] , N 7 15 ; M #000 , S 15 [ 16 9 ] , E 16 + 9 = 25 R [ 25 ] , N 25 15 ; M #00 , S 15 [ 13 13 22 ] , E 22 - 16 = 6 R [ 13 6 ] , G #001 15 [ 13 6 ] , M #001 , S 15 [ 13 6 ] , E 13 + 6 = 19 R [ 19 ] , N 19 15 ; M #001 , S 15 [ 13 6 ] , E 13 - 6 = 7 R [ 7 ] , N 7 15 ; M #00 , S 15 [ 13 13 22 ] , E 22 - 13 = 9 R [ 16 9 ] , G #002 15 [ 16 9 ] , M #002 , S 15 [ 16 9 ] , E 16 - 9 = 7 R [ 7 ] , N 7 15 ; M #002 , S 15 [ 16 9 ] , E 16 + 9 = 25 R [ 25 ] , N 25 15 ; M #00 , S 15 [ 13 13 22 ] , E 13 + 13 = 26 R [ 22 26 ] , G #003 15 [ 22 26 ] , M #003 , S 15 [ 22 26 ] , E 26 - 22 = 4 R [ 4 ] , N 4 15 ; M #00 , S 15 [ 13 13 22 ] , E 13 + 22 = 35 R [ 13 35 ] , G #004 15 [ 13 35 ] , M #004 , S 15 [ 13 35 ] , E 35 - 13 = 22 R [ 22 ] , N 22 15 ; M #00 , S 15 [ 13 13 22 ] , E 13 + 22 = 35 R [ 13 35 ] , G #00